In [1]:
!pip install tensorflow

In [2]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from tensorflow.keras import Input, Model
from tensorflow.keras.layers import Flatten, Dense, Concatenate
from tensorflow.keras.preprocessing.image import load_img, img_to_array

2025-04-22 03:33:10.564328: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-04-22 03:33:13.208309: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-04-22 03:33:13.874826: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1745292794.557034    4965 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1745292794.850228    4965 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1745292797.896420    4965 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linkin

In [3]:
# 1. Load
df = pd.read_csv('youtube_cleaned_success_only.csv')
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11731 entries, 0 to 11730
Data columns (total 12 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   video_id       11731 non-null  object
 1   title          11731 non-null  object
 2   channel        11731 non-null  object
 3   channel_id     11731 non-null  object
 4   published      11731 non-null  object
 5   views          11731 non-null  int64 
 6   likes          11731 non-null  int64 
 7   thumbnail_url  11731 non-null  object
 8   category_id    11731 non-null  int64 
 9   subscribers    11731 non-null  int64 
 10  region         11731 non-null  object
 11  Source.Name    1823 non-null   object
dtypes: int64(4), object(8)
memory usage: 1.1+ MB


In [4]:
df['published']    = pd.to_datetime(df['published'],
                                    format="%Y-%m-%dT%H:%M:%SZ",
                                    errors='coerce')
df.dropna(subset=['published'], inplace=True)
df.reset_index(drop=True, inplace=True)
df['publish_hour'] = df['published'].dt.hour
df['day_of_week']  = df['published'].dt.day_name()

rads = 2 * np.pi * df['publish_hour'] / 24

# 2) create two features
df['hour_sin'] = np.sin(rads)
df['hour_cos'] = np.cos(rads)

# 3) drop the original
df.drop(columns=['publish_hour'], inplace=True)

df.drop(['Source.Name'], axis=1, inplace=True, errors='ignore')


In [5]:
# 3. Build image tensor
IMG_SIZE = (128, 128)
def load_image(video_id):
    path = f'youtube_thumbnails/{video_id}.jpg'
    try:
        img = load_img(path, target_size=IMG_SIZE)
        return img_to_array(img) / 255.0
    except:
        return np.zeros((*IMG_SIZE, 3))

image_tensor = np.stack(df['video_id'].apply(load_image).values)  # shape = (N,128,128,3)

In [6]:
# 4. Prepare tabular features
tab_df = df.drop(columns=[
    'video_id','views','title','channel','channel_id','thumbnail_url',
    'published'
])
tab_df = pd.get_dummies(
    tab_df,
    columns=['region','day_of_week','category_id'],
    drop_first=True
)
scaler = StandardScaler()
X_tabular = scaler.fit_transform(tab_df)  # shape = (N, D_tab)

In [7]:
# 5. Target
y = np.log1p(df['views'].values)

# 6. Train/validation split
X_img_train, X_img_val, X_tab_train, X_tab_val, y_train, y_val = train_test_split(
    image_tensor, X_tabular, y, test_size=0.2, random_state=42
)

In [8]:
from tensorflow.keras.preprocessing.image import load_img, img_to_array
import numpy as np

# 统一尺寸
IMG_SIZE = (128, 128)

def load_image(video_id):
    path = f'youtube_thumbnails/{video_id}.jpg'
    try:
        img = load_img(path, target_size=IMG_SIZE)
        return img_to_array(img) / 255.0  # 归一化
    except:
        return np.zeros((*IMG_SIZE, 3))  # 出错则填0


In [9]:
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, Flatten, Dense, Concatenate
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

In [10]:
# Callback
es = EarlyStopping(patience=5, restore_best_weights=True)

# --- 图像分支 ---
img_in = Input(shape=(128, 128, 3), name='img_input')
c = Conv2D(32, (3,3), activation='relu', padding='same')(img_in)
c = MaxPooling2D((2,2))(c)
c = Conv2D(64, (3,3), activation='relu', padding='same')(c)
c = MaxPooling2D((2,2))(c)
c = Flatten()(c)
c = Dense(32, activation='relu')(c)

# --- 表格分支 ---
tab_in = Input(shape=(X_tab_train.shape[1],), name='tab_input')
t = Dense(32, activation='relu')(tab_in)


2025-04-22 03:33:58.074153: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


In [11]:
# --- 合并 + 输出 ---
f = Concatenate()([c, t])
f = Dense(16, activation='relu')(f)
out = Dense(1, activation='linear')(f)

cnn_model = Model(inputs=[img_in, tab_in], outputs=out)
cnn_model.compile(optimizer=Adam(1e-3), loss='mse')


In [12]:
# --- 训练 ---
cnn_model.fit(
    [X_img_train, X_tab_train], y_train,
    validation_data=([X_img_val, X_tab_val], y_val),
    epochs=20,
    batch_size=32,
    callbacks=[es],
    verbose=2
)

Epoch 1/20
248/248 - 52s - 209ms/step - loss: 9.9750 - val_loss: 5.7956
Epoch 2/20
248/248 - 48s - 195ms/step - loss: 4.1675 - val_loss: 4.4378
Epoch 3/20
248/248 - 49s - 196ms/step - loss: 3.4683 - val_loss: 3.7538
Epoch 4/20
248/248 - 49s - 196ms/step - loss: 3.0430 - val_loss: 3.6896
Epoch 5/20
248/248 - 49s - 197ms/step - loss: 2.8012 - val_loss: 3.4929
Epoch 6/20
248/248 - 48s - 195ms/step - loss: 2.2001 - val_loss: 3.3481
Epoch 7/20
248/248 - 49s - 196ms/step - loss: 1.9693 - val_loss: 3.4162
Epoch 8/20
248/248 - 49s - 196ms/step - loss: 1.5862 - val_loss: 3.3110
Epoch 9/20
248/248 - 49s - 196ms/step - loss: 1.3531 - val_loss: 3.3477
Epoch 10/20
248/248 - 49s - 196ms/step - loss: 1.1846 - val_loss: 3.6706
Epoch 11/20
248/248 - 48s - 193ms/step - loss: 0.9692 - val_loss: 3.6061
Epoch 12/20
248/248 - 49s - 196ms/step - loss: 0.8075 - val_loss: 3.5960
Epoch 13/20
248/248 - 49s - 196ms/step - loss: 0.7135 - val_loss: 3.5907


In [14]:
# --- 预测与评估 ---
y_pred_log = cnn_model.predict([X_img_val, X_tab_val])

# --- 简易评估函数 ---
def report_metrics(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    print(f"MAE : {mae:.4f}")
    print(f"RMSE: {rmse:.4f}")
    print(f"R²  : {r2:.4f}")

print("=== Small CNN Multi‑Input ===")
report_metrics(y_val, y_pred_log)

62/62 ━━━━━━━━━━━━━━━━━━━━ 3s 53ms/step
=== Small CNN Multi‑Input ===
MAE : 1.1987
RMSE: 1.8196
R²  : 0.6490
